In [2]:
!pip install -q imagehash opencv-python-headless tqdm kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 1.8 MB/s eta 0:00:00


In [3]:
# ── Environment Setup ──────────────────────────────────────────────────────
import os
import json
import logging
from pathlib import Path
import sys

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s', stream=sys.stdout, force=True)

os.environ['KAGGLE_USERNAME'] = "paranietharan29"
os.environ['KAGGLE_KEY']      = "KGAT_21968fa3538f832e53f6adc53b096497"

# ── Paths ─────────────────────────────────────────────────────────────────────
RAW_DATA_PATH  = Path("/content/raw_data")
DATASET_PATH   = Path("/content/dataset")
QUARANTINE_PATH = Path("/content/quarantine")

for p in [RAW_DATA_PATH, DATASET_PATH, QUARANTINE_PATH]:
    p.mkdir(parents=True, exist_ok=True)

logging.info("Environment setup complete.")

INFO: Environment setup complete.


In [4]:
# ── Imports ────────────────────────────────────────────────────────────────
import zipfile
import shutil
import numpy as np
import imagehash
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split

INFO: NumExpr defaulting to 2 threads.


In [5]:
import torch
from pathlib import Path

# TRAINING CONFIGURATION
MODEL_TYPE = 'yolov8n.pt' # 'yolov8n.pt' for speed, 'yolov8s.pt' for better accuracy
EPOCHS = 50
PATIENCE = 10 # Early stopping patience
IMG_SIZE = 640
BATCH_SIZE = 16

In [6]:
import os
import json
import logging
from pathlib import Path
import sys
import zipfile
import shutil
from google.colab import drive # Import drive

# ── Download Dataset ───────────────────────────────────────────────────────
def download_dataset():
    logging.info("entering into download dataset")

    # 1. Mount Google Drive
    drive.mount('/content/drive', force_remount=True) # force_remount=True ensures a clean mount
    DRIVE_DATASET_FOLDER_NAME = "RDD2022_Dataset_Raw" # Folder within MyDrive to store raw dataset
    drive_rdd_split_path = Path("/content/drive/MyDrive") / DRIVE_DATASET_FOLDER_NAME / "RDD_SPLIT"
    local_rdd_split_path = RAW_DATA_PATH / "RDD_SPLIT" # The folder that Kaggle download extracts to

    # 2. Check if dataset already exists in Google Drive
    if drive_rdd_split_path.exists() and any(drive_rdd_split_path.iterdir()):
        logging.info(f"Dataset (RDD_SPLIT) found in Google Drive at {drive_rdd_split_path}.")
        logging.info(f"Copying from Drive to local {local_rdd_split_path}...")

        # Ensure local destination is clean before copying
        if local_rdd_split_path.exists():
            shutil.rmtree(local_rdd_split_path)
        local_rdd_split_path.parent.mkdir(parents=True, exist_ok=True) # Ensure RAW_DATA_PATH exists
        shutil.copytree(drive_rdd_split_path, local_rdd_split_path)
        logging.info("Dataset copied from Drive to local successfully. Skipping Kaggle download.")
        return # Exit early as dataset is ready

    else:
        logging.info("Dataset (RDD_SPLIT) not found in Google Drive. Proceeding with Kaggle download.")

    # Existing Kaggle download logic
    zip_path = RAW_DATA_PATH / "rdd-2022.zip"
    existing_folders = [p for p in RAW_DATA_PATH.iterdir() if p.is_dir()]
    if existing_folders:
        logging.info(f"Local existing folders found: {existing_folders}. Deleting them.")
        for folder in existing_folders:
            shutil.rmtree(folder)

    # Ensure RAW_DATA_PATH exists before downloading
    RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)

    logging.info("Downloading RDD-2022 dataset (~9.9 GB) from Kaggle...")
    kaggle_download_command = f"kaggle datasets download -d aliabdelmenam/rdd-2022 -p {RAW_DATA_PATH}"
    os.system(kaggle_download_command)

    if zip_path.exists():
        logging.info("Extracting zip file...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(RAW_DATA_PATH)
        zip_path.unlink()  # delete zip after extraction to free space
        logging.info("Extraction complete. Zip deleted to save space.")

        # After successful download and extraction, save to Google Drive for future use
        if local_rdd_split_path.exists(): # This should now be true if extraction was successful
            logging.info(f"Dataset downloaded and extracted locally. Saving to Google Drive at {drive_rdd_split_path} for future fast loading.")
            drive_rdd_split_path.parent.mkdir(parents=True, exist_ok=True) # Ensure parent dir in Drive exists

            # If the drive path exists, remove it to ensure a fresh copy
            if drive_rdd_split_path.exists():
                shutil.rmtree(drive_rdd_split_path)
            shutil.copytree(local_rdd_split_path, drive_rdd_split_path)
            logging.info("Dataset successfully saved to Google Drive.")
        else:
            logging.warning("Local RDD_SPLIT folder not found after extraction. Skipping save to Drive.")

    else:
        logging.error("Kaggle download failed — zip file not found. Check Kaggle credentials and internet connection.")

In [7]:
# ── VOC → YOLO Conversion ──────────────────────────────────────────────────
# RDD-2022 damage classes: D00=longitudinal crack, D10=transverse crack,
# D20=alligator crack, D40=pothole. All mapped to class 0 ('damage').
TARGET_CLASS_MAP = {"D00": 0, "D10": 0, "D20": 0, "D40": 0}

def convert_voc_to_yolo(xml_file, target_class_map=None):
    """Parse a PascalVOC XML and return YOLO-format label strings.

    FIX: uses explicit named variables for coordinates instead of a
    positional tuple (b[0]..b[3]) which was error-prone and confusing.
    The math is identical but now self-documenting and safe.
    """
    if target_class_map is None:
        target_class_map = TARGET_CLASS_MAP

    try:
        tree = ET.parse(xml_file)
    except ET.ParseError as e:
        logging.warning(f"Malformed XML skipped: {xml_file} — {e}")
        return []

    root = tree.getroot()
    size = root.find('size')
    if size is None:
        return []

    w = int(size.find('width').text)
    h = int(size.find('height').text)
    if w == 0 or h == 0:
        return []

    yolo_labels = []
    for obj in root.iter('object'):
        cls_name = obj.find('name').text
        if cls_name not in target_class_map:
            continue

        cls_id = target_class_map[cls_name]
        xmlbox = obj.find('bndbox')

        # FIX: explicit names — no more confusing b[0]..b[3] indexing
        xmin = float(xmlbox.find('xmin').text)
        xmax = float(xmlbox.find('xmax').text)
        ymin = float(xmlbox.find('ymin').text)
        ymax = float(xmlbox.find('ymax').text)

        # YOLO format: x_center y_center width height (all normalised 0–1)
        x_center = (xmin + xmax) / (2.0 * w)
        y_center = (ymin + ymax) / (2.0 * h)
        bbox_w   = (xmax - xmin) / w
        bbox_h   = (ymax - ymin) / h

        # Clamp to [0, 1] to guard against annotation errors
        x_center = max(0.0, min(1.0, x_center))
        y_center = max(0.0, min(1.0, y_center))
        bbox_w   = max(0.0, min(1.0, bbox_w))
        bbox_h   = max(0.0, min(1.0, bbox_h))

        yolo_labels.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {bbox_w:.6f} {bbox_h:.6f}")

    return yolo_labels

In [8]:
def clean_and_process():
    """
    The Kaggle aliabdelmenam/rdd-2022 dataset is already in YOLO format.
    Labels are .txt files in train/labels/ alongside train/images/.
    Just validate image integrity and pair each image with its label.
    """
    split_root = RAW_DATA_PATH / "RDD_SPLIT"
    image_files = list((split_root / "train" / "images").glob("*.jpg"))
    print(f"Found {len(image_files)} images")

    if not image_files:
        logging.error("No images found in RDD_SPLIT/train/images/")
        return []

    valid_samples = []
    stats = {'integrity_fail': 0, 'no_label': 0, 'empty_label': 0}

    for img_path in tqdm(image_files):
        # Integrity check
        try:
            with Image.open(img_path) as im:
                im.verify()
        except Exception:
            stats['integrity_fail'] += 1
            continue

        # Find paired .txt label file
        label_path = split_root / "train" / "labels" / img_path.with_suffix(".txt").name
        if not label_path.exists():
            stats['no_label'] += 1
            continue

        # Read existing YOLO labels — filter to class 0 only (damage)
        lines = [l.strip() for l in label_path.read_text().splitlines() if l.strip()]
        labels = [l for l in lines if l.startswith("0 ")]  # keep class 0 only

        if not labels:
            stats['empty_label'] += 1
            continue

        valid_samples.append({'img': img_path, 'labels': labels})

    logging.info(f"Done. Valid: {len(valid_samples)}, Stats: {stats}")
    return valid_samples

In [9]:
# ── Dataset Splitting & YOLO Config ───────────────────────────────────────
def finalize_dataset(samples):
    if not samples:
        logging.error("Cannot finalize dataset: samples list is empty!")
        return

    train, temp = train_test_split(samples, test_size=0.30, random_state=42)
    val,   test = train_test_split(temp,    test_size=0.33, random_state=42)

    logging.info(f"Split sizes — train: {len(train)}, val: {len(val)}, test: {len(test)}")

    splits = {'train': train, 'val': val, 'test': test}
    for split_name, data in splits.items():
        img_dir   = DATASET_PATH / split_name / 'images'
        label_dir = DATASET_PATH / split_name / 'labels'
        img_dir.mkdir(parents=True, exist_ok=True)
        label_dir.mkdir(parents=True, exist_ok=True)

        for item in data:
            shutil.copy(item['img'], img_dir / item['img'].name)
            label_path = label_dir / f"{item['img'].stem}.txt"
            with open(label_path, 'w') as f:
                f.write("\n".join(item['labels']))

    # data.yaml — using absolute paths (required for Colab/YOLO)
    yaml_content = (
        f"path: {DATASET_PATH}\n"
        f"train: train/images\n"
        f"val:   val/images\n"
        f"test:  test/images\n"
        f"nc: 1\n"
        f"names: ['road_damage']\n"
    )
    yaml_path = DATASET_PATH / 'data.yaml'
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)

    logging.info(f"Dataset finalised. YAML written to {yaml_path}")

In [ ]:
download_dataset()

INFO: entering into download dataset
Mounted at /content/drive
INFO: Dataset (RDD_SPLIT) found in Google Drive at /content/drive/MyDrive/RDD2022_Dataset_Raw/RDD_SPLIT.
INFO: Copying from Drive to local /content/raw_data/RDD_SPLIT...


In [ ]:
# ── Clean & Process
valid_data = clean_and_process()

Found 26869 images


100%|██████████| 26869/26869 [00:27<00:00, 972.03it/s] 

INFO: Done. Valid: 9457, Stats: {'integrity_fail': 0, 'no_label': 0, 'empty_label': 17412}


In [ ]:
# ── Finalize Dataset
finalize_dataset(valid_data)

INFO: Split sizes — train: 6619, val: 1901, test: 937
INFO: Dataset finalised. YAML written to /content/dataset/data.yaml


In [ ]:
import torch
device = "0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: 0


In [ ]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.7 MB/s eta 0:00:00


In [ ]:
import torch
from ultralytics import YOLO
from pathlib import Path

# ── Train YOLO Model (Restoring device and hyperparameters) ────────────────
# Using configuration from cell 10ba6e98

# Initialize model
model = YOLO(MODEL_TYPE)

# Start training with full parameterization
results = model.train(
    data=str(DATASET_PATH / "data.yaml"),
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device="0" if torch.cuda.is_available() else "cpu",
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,
    augment=True,
    cache=False,
    workers=2,
    project="/content/runs",
    name="rdd2022_yolov8_optimized",
    exist_ok=True
)

# Capture best weights path
best_weights = Path(results.save_dir) / 'weights' / 'best.pt'
print(f"\n✅ Training complete. Best weights: {best_weights}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.49 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640

In [ ]:
# ── Validate on val set ───────────────────────────────────────────────────
metrics = model.val()
print("mAP50:    ", metrics.box.map50)
print("mAP50-95: ", metrics.box.map)

In [ ]:
# ── Quick inference on a test image ──────────────────────────────────────
from ultralytics import YOLO
import glob

best_weights = "/content/runs/rdd2022_yolov8n/weights/best.pt"
model = YOLO(best_weights)

test_images = glob.glob(str(DATASET_PATH / "test/images/*.jpg"))[:5]
results = model.predict(test_images, conf=0.25, save=True, project="/content/runs", name="predict")
print("Saved predictions to /content/runs/predict/")

In [ ]:
from google.colab import drive
import shutil
from pathlib import Path

def save_model_to_drive(model_source_path, drive_folder_name="RDD2022_YOLO_Models"):
    logging.info("Mounting Google Drive...")
    drive.mount('/content/drive')

    drive_dest_dir = Path(f"/content/drive/MyDrive/{drive_folder_name}")
    drive_dest_dir.mkdir(parents=True, exist_ok=True)

    model_filename = Path(model_source_path).name
    dest_path = drive_dest_dir / model_filename

    if Path(model_source_path).exists():
        logging.info(f"Copying model from {model_source_path} to {dest_path}")
        shutil.copy(model_source_path, dest_path)
        logging.info(f"Model saved to Google Drive: {dest_path}")
    else:
        logging.error(f"Model file not found at {model_source_path}. Please ensure training completed successfully and the path is correct.")


In [ ]:
save_model_to_drive(best_weights)